# 05 — Synthetic benchmark v0.1: pilot generation walkthrough

Runs the reusable generator package (`src/boamp/synthetic/`) end-to-end for
one scenario and inspects the result. All generation logic lives in
`src/boamp/synthetic/*.py`; this notebook only calls it and displays outputs
(spec: "Do not place core generation logic only in the notebook").

Adapts the gold-standard-and-corruption framework of Lam et al. (2024) to
public-procurement recurrence linkage: a clean latent world (buyers,
establishments, needs, cycles, true relations) is generated first, then
BOAMP-like publication notices are corrupted into `observed_notices`, while
the truth tables are retained separately and never exposed to Layer 1/Layer 2.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from boamp.synthetic.parameters import load_calibration_parameters
from boamp.synthetic.scenarios import VALID_SCENARIOS, load_benchmark_defaults, load_scenario
from boamp.synthetic.pipeline import generate_clean_world, generate_observed_world, write_pilot_outputs

pd.set_option("display.max_colwidth", 120)
print("Available scenarios:", VALID_SCENARIOS)

Available scenarios: ('clean_sanity', 'central_provisional', 'adverse_identity')


## 1. Load configuration and select a scenario

In [2]:
SCENARIO_ID = "central_provisional"

calib = load_calibration_parameters(PROJECT_ROOT)
defaults = load_benchmark_defaults(PROJECT_ROOT)
scenario = load_scenario(PROJECT_ROOT, SCENARIO_ID)

print("benchmark_id:", defaults.benchmark_id)
print("target_n_buyers:", defaults.target_n_buyers, " target_n_notices (approx):", defaults.target_n_notices)
print("seeds:", defaults.seed.latent_world_seed, defaults.seed.corruption_seed)
print()
print(f"Scenario: {scenario.display_name!r} ({scenario.scenario_id})")
print("Purpose:", scenario.purpose.strip())
print("base_recurrence_propensity:", scenario.recurrence.base_recurrence_propensity)
print("identifiers.siret_missing_rate:", scenario.identifiers.siret_missing_rate)
print("cpv.missing_rate:", scenario.cpv.missing_rate)

benchmark_id: synthetic_benchmark_v0.1_provisional
target_n_buyers: 2000  target_n_notices (approx): 10000
seeds: 20260721 20260722

Scenario: 'Central provisional' (central_provisional)
Purpose: Principal v0.1 benchmark. Uses calibrated BOAMP observation mechanisms (identifier/CPV/duration/text missingness and buyer-name variation) at their observed real-corpus rates; recurrence prevalence, cycle-gap distribution, and text drift remain explicit scenario assumptions, never calibrated to a point (see calib_unidentified_parameters_scenarios.csv).
base_recurrence_propensity: 0.4
identifiers.siret_missing_rate: 0.728
cpv.missing_rate: 0.1706


## 2. Run a small pilot (n_buyers below the full 2,000-buyer default, for a fast notebook run)

In [3]:
world = generate_clean_world(SCENARIO_ID, PROJECT_ROOT, n_buyers=300, world_seed=defaults.seed.latent_world_seed)
print("Clean-world structural validation:", "PASS" if world["_validation"].passed else world["_validation"].failures())

observed, corruption_log = generate_observed_world(world, SCENARIO_ID, PROJECT_ROOT, corruption_seed=defaults.seed.corruption_seed)
print("observed_notices rows:", len(observed), " corruption_log rows:", len(corruption_log))

Clean-world structural validation: PASS


observed_notices rows: 1311  corruption_log rows: 4496


## 3. Row counts

In [4]:
row_counts = {k: len(v) for k, v in world.items() if k != "_validation"}
row_counts["observed_notices"] = len(observed)
row_counts["corruption_log"] = len(corruption_log)
pd.Series(row_counts, name="n_rows").to_frame()

,n_rows
buyers,300
establishments,325
needs,702
cycles,957
true_relations,957
notice_family_membership,1311
clean_notices,1311
observed_notices,1311
corruption_log,4496


## 4. Structural checks

In [5]:
from boamp.synthetic.validation import run_full_structural_validation

result = run_full_structural_validation(world)
pd.Series(result.checks, name="passed").to_frame()

,passed
all_siren_valid,True
all_siret_valid,True
siret_belongs_to_siren,True
every_establishment_has_one_buyer,True
every_need_has_one_buyer,True
every_need_has_one_establishment,True
every_cycle_has_one_need,True
every_cycle_has_one_buyer,True
valid_relation_types,True
exactly_one_outgoing_edge_per_cycle,True


## 5. Clean -> corrupted examples

In [6]:
clean_idx = world["clean_notices"].set_index("notice_id_synthetic")
sample_ids = observed["notice_id_synthetic"].sample(5, random_state=1).tolist()

compare_cols = ["buyer_siret_raw", "buyer_siren_raw", "buyer_name_raw", "cpv_clean",
                "declared_duration_months", "objet_clean", "linked_call_notice_id"]
clean_compare_cols = ["siret_true", "siren_true", "buyer_name_true", "cpv_true",
                      "duration_true_months", "objet_true", "linked_call_notice_id_true"]

rows = []
for nid in sample_ids:
    c = clean_idx.loc[nid]
    o = observed[observed["notice_id_synthetic"] == nid].iloc[0]
    for cc, oc in zip(clean_compare_cols, compare_cols):
        rows.append({"notice_id": nid, "field": oc, "clean": c[cc], "observed": o[oc]})
pd.DataFrame(rows)

,notice_id,field,clean,observed
0,NOTICE-00000201,buyer_siret_raw,63730765300013,None
1,NOTICE-00000201,buyer_siren_raw,637307653,None
2,NOTICE-00000201,buyer_name_raw,Syndicat mixte Fonchamp,Collectivité territoriale
3,NOTICE-00000201,cpv_clean,45162976,None
4,NOTICE-00000201,declared_duration_months,15.913028,NaN
5,NOTICE-00000201,objet_clean,Marché public de construction d'école - Syndicat mixte Fonchamp - lot 2 : réhabilitation et construction.,"Prestations diverses dans le cadre d'un marché public, se reporter au dossier de consultation."
6,NOTICE-00000201,linked_call_notice_id,None,None
7,NOTICE-00000115,buyer_siret_raw,09594272800015,09594272800015
8,NOTICE-00000115,buyer_siren_raw,095942728,095942728
9,NOTICE-00000115,buyer_name_raw,Commune de Montbourg,Commune de Montbourg


## 6. Corruption type summary

In [7]:
corruption_log["corruption_type"].value_counts().to_frame("n_events")

,n_events
corruption_type,
BOTH_MISSING,1434
MISSING,1272
GENERIC_BOILERPLATE,598
GENERIC_NAME_COLLISION,277
LINK_NOT_RECORDED,207
SIREN_ONLY_OBSERVED,194
NAME_VARIANT,125
ROUNDED_OR_CONVERTED,101
GENERIC,98


In [8]:
corruption_log.groupby("field")["corruption_type"].value_counts().unstack(fill_value=0)

corruption_type,BOTH_MISSING,DIVISION_ONLY,GENERIC,GENERIC_BOILERPLATE,GENERIC_NAME_COLLISION,INVALID_CHECKSUM,LEXICAL_DRIFT,LINK_NOT_RECORDED,MISSING,NAME_VARIANT,PARENT,ROUNDED_OR_CONVERTED,SIREN_ONLY_OBSERVED,WRONG_ESTABLISHMENT_SAME_SIREN,WRONG_RELATED
field,,,,,,,,,,,,,,,
buyer_name_raw,0,0,0,0,277,0,0,0,0,125,0,0,0,0,0
buyer_siren_raw,717,0,0,0,0,0,0,0,0,0,0,0,0,0,0
buyer_siret_raw,717,0,0,0,0,13,0,0,0,0,0,0,194,3,0
cpv_clean,0,27,98,0,0,0,0,0,193,0,56,0,0,0,31
declared_duration_months,0,0,0,0,0,0,0,0,1079,0,0,101,0,0,0
linked_call_notice_id,0,0,0,0,0,0,0,207,0,0,0,0,0,0,0
objet_clean,0,0,0,598,0,0,60,0,0,0,0,0,0,0,0


## 7. Recurrence (NEXT_CYCLE) and NO_SUCCESSOR examples

In [9]:
next_cycle_examples = world["true_relations"][world["true_relations"]["relation_type"] == "NEXT_CYCLE"].head(5)
next_cycle_examples[["source_cycle_id", "target_cycle_id", "true_gap_months", "source_expected_end", "target_start"]]

,source_cycle_id,target_cycle_id,true_gap_months,source_expected_end,target_start
2,CYCLE-00000002,CYCLE-00000003,2.201051,2025-04-06,2025-06-12
8,CYCLE-00000008,CYCLE-00000009,16.425756,2019-07-12,2020-11-23
9,CYCLE-00000009,CYCLE-00000010,10.512484,2021-07-13,2022-05-29
10,CYCLE-00000010,CYCLE-00000011,13.074901,2023-01-03,2024-02-05
20,CYCLE-00000020,CYCLE-00000021,28.909330,2023-05-22,2025-10-18


In [10]:
no_successor_examples = world["true_relations"][world["true_relations"]["relation_type"] == "NO_SUCCESSOR"].head(5)
no_successor_examples[["source_cycle_id", "target_cycle_id", "source_expected_end"]]

,source_cycle_id,target_cycle_id,source_expected_end
0,CYCLE-00000000,None,2021-02-18
1,CYCLE-00000001,None,2023-05-23
3,CYCLE-00000003,None,2026-07-25
4,CYCLE-00000004,None,2024-10-25
5,CYCLE-00000005,None,2019-11-23


In [11]:
print("Relation type mix:")
print(world["true_relations"]["relation_type"].value_counts(normalize=True))
gaps = world["true_relations"].loc[world["true_relations"]["relation_type"] == "NEXT_CYCLE", "true_gap_months"]
print(f"\ntrue_gap_months beyond the real pipeline's 6-month window: {(gaps > 6).mean():.1%} of NEXT_CYCLE edges")

Relation type mix:
relation_type
NO_SUCCESSOR    0.733542
NEXT_CYCLE      0.266458
Name: proportion, dtype: float64

true_gap_months beyond the real pipeline's 6-month window: 87.5% of NEXT_CYCLE edges


## 8. Full pilot outputs

The full 2,000-buyer pilot for all three scenarios (`clean_sanity`,
`central_provisional`, `adverse_identity`) was already generated via
`boamp.synthetic.pipeline.generate_pilot` and is on disk at:

`data/processed/synthetic_benchmark/v0_1_provisional/<scenario>/world_001/corruption_001/`

Each directory contains `latent_buyers.parquet`, `latent_establishments.parquet`,
`latent_needs.parquet`, `latent_cycles.parquet`, `true_relations.parquet`,
`notice_family_membership.parquet`, `clean_notices.parquet`,
`observed_notices.parquet`, `corruption_log.parquet`, and
`generation_metadata.json`. Structural validation results for all three are
in `reports/generated/synthetic_benchmark/v0_1_clean_world_validation.md`.
Fidelity validation against the real corpus is in
`06_synthetic_fidelity_validation.ipynb`.


In [12]:
import json
for scen in ["clean_sanity", "central_provisional", "adverse_identity"]:
    meta_path = PROJECT_ROOT / "data/processed/synthetic_benchmark/v0_1_provisional" / scen / "world_001/corruption_001/generation_metadata.json"
    meta = json.loads(meta_path.read_text())
    print(scen, "-> validation:", meta["validation_status"], " row_counts:", meta["row_counts"])

clean_sanity -> validation: PASS  row_counts: {'buyers': 2000, 'establishments': 2174, 'needs': 4893, 'cycles': 12794, 'true_relations': 12794, 'notice_family_membership': 17708, 'clean_notices': 17708, 'observed_notices': 17708, 'corruption_log': 3683}
central_provisional -> validation: PASS  row_counts: {'buyers': 2000, 'establishments': 2174, 'needs': 4893, 'cycles': 6914, 'true_relations': 6914, 'notice_family_membership': 9537, 'clean_notices': 9537, 'observed_notices': 9537, 'corruption_log': 32710}
adverse_identity -> validation: PASS  row_counts: {'buyers': 2000, 'establishments': 2174, 'needs': 4893, 'cycles': 6914, 'true_relations': 6914, 'notice_family_membership': 9567, 'clean_notices': 9567, 'observed_notices': 9567, 'corruption_log': 39773}
